# 2. Label-free sub-region estimation and the seventeen volumetric features

Reproduces **Section 3.2** (sub-region estimation), **Section 3.3** (the features) and
**Section 4.2** (`fig:subregions`, drawn in notebook 7).

Stages two and three of the pipeline. Neither uses a ground-truth label of any kind: both work
from the whole-tumor mask that stage one predicted.

## The two thresholds

The enhancement map is the voxel-wise difference between the post-contrast and pre-contrast
scans, $\mathrm{enh} = \mathrm{T1ce} - \mathrm{T1}$, which is bright wherever tissue takes up
contrast. Its mean $\mu$ and standard deviation $\sigma$ are measured over normal-appearing
brain outside the predicted tumor, which gives a reference for healthy tissue. Inside the mask:

$$\mathrm{ET} = \{\,\mathrm{enh} > \mu + 1.5\sigma\,\},\qquad
  \mathrm{NCR} = \{\,\mathrm{T1ce} < \mu_t - 0.5\sigma_t\,\}$$

where $\mu_t$ and $\sigma_t$ are the mean and standard deviation of T1ce over the same healthy
region. Enhancing tumor takes precedence over necrotic core, and the rest of the mask is edema.
These two thresholds replace the sub-region labels other methods need.

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

from src import config
print(config.describe())

In [ ]:
from src import subregions, cohort, figures

subregions.self_test()

The self-test above asserts the properties the rest of the pipeline depends on, with no data:
the three sub-regions cover the mask exactly, they are mutually disjoint, an empty mask gives
empty sub-regions, and a feature vector holds seventeen finite numbers.

## The seventeen features

Four volumes, four sub-region fractions of the tumor volume, then mean intensities on T1ce, on
the enhancement map and on FLAIR, two of them normalized by healthy tissue. None of them use
texture, which is what lets Section 4.3 conclude that the grading signal is volumetric.

In [ ]:
figures.feature_table()

## One case, end to end

With the BraTS copy in place, this walks a single case through the two thresholds and prints the
resulting compartment sizes and feature vector. The high-grade case shown is the one the pipeline
figure uses.

In [ ]:
CASE = "BraTS20_Training_045"
HAVE_VOLUMES = config.NPZ_DIR.exists()

if HAVE_VOLUMES:
    volumes = figures.load_case(CASE)
    ET, NCR, ED = subregions.estimate_subregions(
        volumes["wt"], volumes["t1ce"], volumes["t1"], volumes["flair"])
    counts = pd.Series({
        "whole tumor": int(volumes["wt"].sum()),
        "enhancing (ET)": int(ET.sum()),
        "necrotic core (NCR)": int(NCR.sum()),
        "edema (ED)": int(ED.sum()),
    }, name="voxels")
    display(counts.to_frame())
    print("the three compartments partition the mask:",
          bool(np.array_equal(ET | NCR | ED, volumes["wt"])))

In [ ]:
if HAVE_VOLUMES:
    vector = subregions.features(ET, NCR, ED, volumes["wt"],
                                 volumes["t1ce"], volumes["t1"], volumes["flair"])
    display(pd.Series(vector, index=subregions.FEATURE_NAMES,
                      name=CASE).to_frame().round(3))

## The cohort feature matrices

Two matrices are built with the same `features` function and differ only in where their
sub-regions come from:

- **predicted**, the sub-regions estimated by the two thresholds above, over 369 cases;
- **ground truth**, the sub-regions read straight off the dataset's label codes
  (`seg == 4`, `seg == 1`, `seg == 2`), over the 368 cases that carry a reference mask.

Both ship in `data_cache/`, which is why notebooks 3 to 6 run without the volumes. The cells
below rebuild them from the volumes when they are available.

In [ ]:
if HAVE_VOLUMES:
    cohort.extract_features(force=True)
    cohort.extract_gt_features(force=True)

predicted = cohort.extract_features()
ground_truth = cohort.extract_gt_features()
grades = cohort.load_grade_labels()

print(f"predicted sub-regions: {len(predicted)} cases")
print(f"ground-truth sub-regions: {len(ground_truth)} cases "
      f"(missing: {sorted(set(predicted) - set(ground_truth))})")
print(f"grades: {len(grades)} cases, "
      f"{sum(v == 'HGG' for v in grades.values())} high-grade, "
      f"{sum(v == 'LGG' for v in grades.values())} low-grade")

In [ ]:
X, y, cases = cohort.build_matrix(predicted, grades)
frame = pd.DataFrame(X, index=cases, columns=subregions.FEATURE_NAMES)
frame.insert(0, "grade", np.where(y == 1, "low-grade", "high-grade"))
display(frame.groupby("grade").mean().round(2).T)

The group means already show what the SHAP attributions in notebook 3 confirm: the enhancing and
necrotic measures separate the grades, while the edema fraction runs the other way.